# Quantization for Neural Networks

## Introduction

Quantization refers to techniques for performing computations and storing tensors at lower bit-widths than floating point precision. A quantized model executes some or all of the operations on tensors with integers rather than floating point values. This allows for a more compact model representaion and the use of high performance vectorized operations on many hardware platforms. This technique is in particular useful at the inference time since it saves a lot of inference computation cost without sacrificing too much inference accuracies.

So far, major deep learning frameworks, such as Tensorflow and Pytorch, have supported quantization natively. The users have been using the built-in quantization modules successfully without knowing how it works exactly. In this artical, I would like to elucidate the mathematics of quantization for nn so that the developers would have some ideas about the quantization mechanisms

## 📎 Quantization

### > Quantization Mapping

Quantizaton maps a floating point value $x \in [\alpha, \beta]$ to  a,b-bit integer $x_q \in [\alpha_q,\beta_q]$.

Mathematically, the $\text{de-quantization}$ process is defined as:
$$
x = c(x_q + d)
$$

and the $\text{quantization}$ process is defined as:
$$
x_q = round(\frac{1}{c}x - d)
$$

where $c,d$ are variables.

In order to derive $c,d$. We have to make sure that $\alpha$ maps to $\alpha_q$ and $\beta$ maps to $\beta_q$. So we would just have to solve the linear system:
$$
\beta = c(\beta_q + d)
$$
$$
\alpha = c(\alpha_q + d)
$$

The solution is:
$$
c = \frac{\beta - \alpha}{\beta_q - \alpha_q}
$$
$$
d = \frac{\alpha.\beta_q - \beta.\alpha_q}{\beta - \alpha}
$$

In practice, we would have to ensure that $0$ in floating point is represented exactly with no error after quantization.

Mathematically, we need to ensure:
$$
x_q = round(\frac{1}{c}0 - d) \\
= round(-d) \\
= -round(d) \\
= -d
$$

This means that:
$$
d = round(d) \\
= round(\frac{\alpha.\beta_q - \beta.\alpha_q}{\beta - \alpha})
$$

By convection, we denote:
- $c$ as the scale ($s$).
- $-d$ as the zero point ($z$).

To summarize, the $\text{de-quantization}$ process is defined as:
$$
x = s(x_q - z)
$$

and the $\text{quantization}$ is defined as:
$$
x_q = round(\frac{1}{s}x + z)
$$

The value of scale $s$ and zero point $z$ are:
$$
s = \frac{\beta - \alpha}{\beta_q - \alpha_q}
$$
$$
z = round(\frac{\alpha.\beta_q - \beta.\alpha_q}{\beta - \alpha})
$$

Note that $z$ is an integer and $s$ is a positive floating point number.

### > Value Clipping

In practice, the quantization process will have chance to have $x$ that is outside the range of $[\alpha, \beta]$, thus the qunatized value $x_q$ will also be outside the range of $[\alpha_q, \beta_q]$. If the $integer$ type is:
- signed $\text{INTb}$:
    $$(\alpha_q, \beta_q) = (-2^{b-1}, 2^{b-1} - 1)$$
- unsigned $\text{UINTb}$:
    $$(0, 2^b - 1)$$

programming languages that have fixed type-precisions will clip the values that are outside the range.

More concretely, the quantization process will have an additional clip step:
$$
x_q = \text{clip}(x_q, \alpha_q, \beta_q)
$$

where:
- $x_q = round(\frac{1}{s}x + z)$
- $clip(x, l, u)$ function is defined as:
$$
clip(x, l, u) =
\begin{cases}
l & \text{if} \space x < l \\
x & \text{if} \space l \leq x \leq u \\
u & \text{if} \space x > u \\
\end{cases}
$$

### > Affine Quantization Mapping

The quantization mapping we discussed above is also called affine quantization mapping.

### > Scale Quantization Mapping

If the integer type is signed $\text{INTb}$, ($\alpha_q, \beta_q$) = $(-2^{b-1} + 1, 2^{b-1} - 1)$, and we force $z = 0$.

Mathematically, we have:
$$
\alpha_q = -\beta_q
$$
$$
round(\frac{\beta.\alpha_q - \alpha.\beta_q}{\beta - \alpha}) = 0
$$

This resulst in $\alpha = -\beta$. Therefore, we are mapping between the floating point range $[\alpha, -\alpha]$ and the integer range $[\alpha_q, -\alpha_q]$.

Because it is exactly symmetric around $0$, we also call it symmetric quantization mapping.

Note that scale quantization mapping is just a spectial case of the affine quantization mapping, and we have an unused bit in the integer range.

### Summary

The quantization function is defined as:
$$
f_q(x, s, z) = \text{clip}(round(\frac{1}{s}x + z), \alpha_q, \beta_q)
$$

and the de-quantization function is defined as:
$$
f_q(x_q, s, z) = s(x_q - z)
$$

There are other ways of quantization mapping, potentially even non-linear. But we will not elaborate them in this article.

## 📎 Quantized Matrix Multiplication

### > Quantized Matrix Multiplication Mathematics

Suppose we have to perform the matrix multiplication $Y = X.W + b$, where $X \in \mathbb{R}^{m \times p}$, $W \in \mathbb{R}^{p \times n}$, and $b \in \mathbb{R}^n$ resulting in $Y \in \mathbb{R}^{m \times n}$:
$$
Y_{i,j} = b_j + \sum_{k=1}^p{X_{i,k}.W_{k,j}}
$$

We would need to do $p$ floating number multiplications and $p$ floating number additions to compute one single entry in $Y$. To complete the full matrix multiplication, given there are $m,n$ in entries in $Y$, we would need to do $m,p,n$ floating number multiplication and $m,p,n$ floating number additions.

Depending on the floating number precision, such the speed of such floating point matrix multiplication might not be favored. So the question becomes can we complete the same matrix multiplication using quantized values.

Here we apply the de-quantization equation:
$$
Y_{i,j} = b_j + \sum_{k=1}^p{X_{i,k}.W_{k,j}} \\
= s_b(b_{q,j} - z_b) + \sum_{k=1}^p{s_X(X_{q,i,k} - z_X).s_W(W_{q,k,j} - z_W)} \\
= s_b(b_{q,j} - z_b) + s_X.s_W.\sum_{k=1}^p{(X_{q,i,k} - z_X).(W_{q,k,j} - z_W)} \\
= s_b(b_{q,j} - z_b) + s_X.s_W.[(\sum_{k=1}^p{X_{q,i,k}.W_{q,k,j}}) - (z_W.\sum_{k=1}^p{X_{q,i,k}}) - (z_X.\sum_{k=1}^p{W_{q,k,j}}) + p.z_X.z_W] \\
= s_Y(Y_{q,i,j} - z_Y)
$$

where $X_q, W_q, b_q, Y_q$ are the quantized matrix for $X, W, b$ and $Y$. respectively $s_X, s_W, s_b$ and $s_Y$ are the scales for $X, W, b, Y$. respectively $z_X, z_W, z_b, z_Y$ are the zero points for $X, W, b, Y$.

Therefore,
$$
Y_{q,i,j} = z_Y + \frac{s_b}{s_Y}(b_{q,j} - z_b) + \frac{s_X.s_W}{s_Y}.[(\sum_{k=1}^p{X_{q,i,k}.W_{q,k,j}}) - (z_W.\sum_{k=1}^p{X_{q,i,k}}) - (z_X.\sum_{k=1}^p{W_{q,k,j}}) + p.z_X.z_W]
$$

Note that in the above equation the following terms are constants during inference and therefore could be computed offline before inference:
- $z_Y$
- $\frac{s_b}{s_Y}(b_{q,j} - z_b)$
- $z_X.\sum_{k=1}^p{W_{q,k,j}}$
- $p.z_X.z_W$

Term $\sum_{k=1}^p{X_{i,k}.W_{k,j}}$ suggests that we could just do the integer matrix multiplication for $X_q$ and $W_q$. Such integer matrix multiplication could employ special hardware and algorithms, such as NVIDIA Tensor Core and Tensor Core IMMA operations, and runs much faster than conventional integer matrix multiplication.

![](image1.png)

One additional thing to note is that $s_X, s_W, s_Y, z_X, z_W$ and $z_Y$ are floating point and integer constants, instead of variables. So there could be some special compile-time optimizations for those multiplications.

This could be retrieved from the resulting integer matrix from X_q and W_q multiplication, which is much faster than the floating number matrix multiplication for the same sizes.

The significance of such quantized matrix multiplication is that the product integer matrix could be converted back to floating point matrix using the scale and the zero point of the product integer matrix and it is almost numerically equivalent. If we have to do a sequence of matrix multiplications whose inputs and outputs are floating point numbers, for example:

$$
X_1 = X_0.W_0 + b_0 \\
X_2 = X_1.W_1 + b_1 \\
... \\
X_n = X_{n-1}.W_{n-1} + b_{n-1}
$$

We could convert the math to the followings using quantized matrices:
$$
X_{0,q} = f_q(X_{0}, s_{X_0}, z_{X_0})\\
X_{1,q} = f_m(X_{0,q}, W_{0,q}, b_{0,q}, s_{X_0}, z_{X_0}, s_{W_0}, z_{W_0}, s_{b_0}, z_{b_0}, s_{X_1}, z_{X_1})\\
X_{2,q} = f_m(X_{1,q}, W_{1,q}, b_{1,q}, s_{X_1}, z_{X_1}, s_{W_1}, z_{W_1}, s_{b_1}, z_{b_1}, s_{X_2}, z_{X_2})\\
...\\
X_{n,q} = f_m(X_{n-1,q}, W_{n-1,q}, b_{n-1,q}, s_{X_{n-1}}, z_{X_{n-1}}, s_{W_{n-1}}, z_{W_{n-1}}, s_{b_{n-1}}, z_{b_{n-1}}, s_{X_n}, z_{X_n})\\
X_n = f_d(X_{n,q}, s_{X_n}, z_{X_n})
$$

where $f_q$ is the quantization function, $f_m$ is the quantized matrix multiplication function, and $f_d$ is the de-quantization function.